# Digiwise: RAG-Powered Digital Literacy Chatbot

A retrieval-augmented generation (RAG) chatbot designed to provide clear, supportive, and beginner-friendly digital literacy guidance.

The project combines **Meta Llama 3.2 3B Instruct**, **Hugging Face Transformers**, **ChromaDB**, **Sentence Transformers embeddings**, and **Gradio** to retrieve relevant knowledge before generating a response.

> **Notebook purpose:** This notebook documents the working prototype and provides a reproducible path for running the chatbot in Google Colab or another compatible Python environment.

## Key Features

- Retrieval-Augmented Generation (RAG) using ChromaDB
- Meta Llama 3.2 3B Instruct for response generation
- Sentence Transformers embeddings for semantic retrieval
- 4-bit quantization to reduce model memory requirements
- User-specific conversation history stored locally as JSON
- Gradio interface for interactive use
- Security-conscious authentication: 

## Architecture

```text
User Question
     │
     ▼
Query Embedding
     │
     ▼
ChromaDB Vector Search ───► Top-k Relevant Documents
     │                              │
     └──────────────┬───────────────┘
                    ▼
          System Prompt + Context
                    │
                    ▼
          Llama 3.2 3B Instruct
                    │
                    ▼
             Digiwise Response
                    │
                    ▼
          Conversation History
```

## Important Notes Before Running

1. The chatbot expects an existing ChromaDB vector store containing the project's knowledge base.
2. The Llama model may require access approval and authentication through Hugging Face.
3. A compatible GPU is recommended because the language model is loaded with 4-bit quantization.
4. User conversation files are created locally in `USERS/`. Do not commit these files to GitHub.
5. This is a research/demo implementation, not a production deployment.

## 1. Environment Setup

The original development notebook used Google Colab. The setup below keeps Colab support while avoiding unnecessary or repeated package installation.

For a GitHub repository, the same dependencies should also be recorded in a `requirements.txt` file. This notebook intentionally installs only the packages required by the current implementation.

In [ ]:
# Install the project dependencies.
# In a GitHub repository, keep the same dependencies in requirements.txt as well.

%pip install -q \
    transformers \
    sentencepiece \
    torch \
    bitsandbytes \
    accelerate \
    huggingface-hub \
    sentence-transformers \
    langchain-huggingface \
    langchain-chroma \
    chromadb \
    gradio

## 2. Configuration

This section centralizes paths and model settings so that they can be changed in one place rather than being repeated throughout the notebook.

### Vector store location

- In Google Colab, the default path points to the original Google Drive location used by the project.
- Outside Colab, the default is `./data/chromadb_vectorstore`.
- You can override the location with the `VECTOR_STORE_PATH` environment variable.

In [ ]:
import os
from pathlib import Path

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"
EMBEDDING_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
RETRIEVAL_K = 3
MAX_NEW_TOKENS = 2050
TEMPERATURE = 0.7
TOP_P = 0.9
MAX_HISTORY_MESSAGES = 12
USER_DATA_DIR = Path("./USERS")

# Detect Google Colab without making the project depend on Colab.
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if "VECTOR_STORE_PATH" in os.environ:
    VECTOR_STORE_PATH = Path(os.environ["VECTOR_STORE_PATH"])
elif IN_COLAB:
    VECTOR_STORE_PATH = Path("/content/drive/MyDrive/chromadb_vectorstore")
else:
    VECTOR_STORE_PATH = Path("./data/chromadb_vectorstore")

print(f"Environment: {'Google Colab' if IN_COLAB else 'Local/Other'}")
print(f"Vector store: {VECTOR_STORE_PATH}")

## 3. Prepare the Vector Store

The chatbot does not create the knowledge base in this notebook. It expects a previously built ChromaDB vector store at the configured location.

If you are using the original Google Colab setup, the vector store should exist at:

`/content/drive/MyDrive/chromadb_vectorstore`

If you are running locally, place the vector store under:

`./data/chromadb_vectorstore`

or set `VECTOR_STORE_PATH` to the appropriate location.

In [ ]:
# Mount Google Drive only when running in Google Colab.
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

if not VECTOR_STORE_PATH.exists():
    raise FileNotFoundError(
        f"ChromaDB vector store not found at: {VECTOR_STORE_PATH}\n"
        "Provide the existing vector store or update VECTOR_STORE_PATH."
    )

print(f"Vector store found: {VECTOR_STORE_PATH}")

## 4. Hugging Face Authentication

**Never place a Hugging Face access token directly in a notebook or source file.**

The original development notebook contained a token in a comment. That token has been removed from this public-ready version.

The following cell opens the Hugging Face authentication flow without embedding a credential in the notebook. If the model requires gated access, make sure your Hugging Face account has access to the model before running the model-loading section.

In [ ]:
from huggingface_hub import notebook_login

# Run this cell and follow the authentication prompt when required.
# Your token is entered through the authentication UI rather than stored in this notebook.
notebook_login()

## 5. Import Libraries

The imports below use the current dedicated LangChain integration packages for Hugging Face embeddings and ChromaDB rather than the deprecated embedding import used in the original notebook.

In [ ]:
import json
import re

import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

## 6. Load the Language Model

The project uses **Meta Llama 3.2 3B Instruct** for response generation.

### Why 4-bit quantization?

The model is loaded using 4-bit NF4 quantization to reduce GPU memory requirements. This makes the model more practical for experimentation in GPU-enabled environments such as Google Colab.

The compute data type is selected from the available GPU capability where possible. This avoids assuming that every GPU supports the same low-precision format.

In [ ]:
# Select a compute dtype based on the available accelerator.
if torch.cuda.is_available():
    major_capability, _ = torch.cuda.get_device_capability()
    COMPUTE_DTYPE = torch.bfloat16 if major_capability >= 8 else torch.float16
else:
    COMPUTE_DTYPE = torch.float32

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Quantization compute dtype: {COMPUTE_DTYPE}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "A CUDA-compatible GPU is recommended for this 4-bit Llama setup. "
        "Run the notebook in a GPU-enabled environment."
    )

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Loaded model: {MODEL_ID}")

## 7. Load the Embedding Model

The embedding model converts user questions and knowledge-base documents into numerical representations so that semantically similar content can be retrieved from ChromaDB.

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_ID
)

print(f"Loaded embedding model: {EMBEDDING_MODEL_ID}")

## 8. Define the Digiwise RAG Chatbot

The chatbot follows this sequence for each user message:

1. Load the user's recent conversation history.
2. Search ChromaDB for the three most relevant documents.
3. Add the retrieved content to the system prompt as RAG context.
4. Send the system prompt and conversation history to Llama.
5. Extract the assistant's response.
6. Save the updated conversation history locally.

The implementation also validates user IDs before using them in filenames and limits the amount of previous conversation sent to the model.

In [ ]:
class DigiwiseChatbot:
    """RAG-powered digital literacy chatbot with local conversation history."""

    def __init__(self, model, tokenizer, embedding, prompt, vector_store_path):
        self.model = model
        self.tokenizer = tokenizer
        self.embedding = embedding
        self.prompt = prompt
        self.vector_store_path = Path(vector_store_path)
        self.directory = USER_DATA_DIR
        self.directory.mkdir(parents=True, exist_ok=True)

        # Connect to the existing ChromaDB vector store.
        self.vectordb = Chroma(
            persist_directory=str(self.vector_store_path),
            embedding_function=self.embedding,
        )

        # Initialize the Hugging Face text-generation pipeline.
        self.pipeline = pipeline(
            "text-generation",
            model=self.model,
            tokenizer=self.tokenizer,
        )

    @staticmethod
    def _safe_user_id(user_id):
        """Allow only safe characters before using a user ID in a filename."""
        user_id = str(user_id).strip()
        if not user_id:
            raise ValueError("User ID cannot be empty.")

        safe_id = re.sub(r"[^A-Za-z0-9_-]", "_", user_id)
        return safe_id[:100]

    def _history_path(self, user_id):
        safe_id = self._safe_user_id(user_id)
        return self.directory / f"{safe_id}_conversation.json"

    def load_history(self, user_id):
        """Load conversation history for a user, or return an empty history."""
        file_path = self._history_path(user_id)

        if file_path.exists():
            try:
                with file_path.open("r", encoding="utf-8") as file:
                    history = json.load(file)
                return history if isinstance(history, list) else []
            except (json.JSONDecodeError, OSError):
                return []

        return []

    def save_history(self, user_id, history):
        """Save conversation history for a user as UTF-8 JSON."""
        file_path = self._history_path(user_id)
        with file_path.open("w", encoding="utf-8") as file:
            json.dump(history, file, ensure_ascii=False, indent=2)

    def system_prompt(self, document):
        """Build the system prompt using retrieved RAG context."""
        return f"""
{self.prompt}

Document Retrieval (RAG):
Use the retrieved document content when it is relevant to the user's digital literacy question.
Do not mention the retrieval process or quote this instruction to the user.

Retrieved context:
{document}
""".strip()

    def chat_function(self, message, user_id):
        """Retrieve relevant context, generate a response, and save the conversation."""
        if not message or not str(message).strip():
            raise ValueError("Message cannot be empty.")

        message = str(message).strip()
        history = self.load_history(user_id)

        # Keep only the most recent messages to control prompt size and memory use.
        history = history[-MAX_HISTORY_MESSAGES:]

        # Retrieve the most relevant knowledge-base documents.
        documents = self.vectordb.similarity_search(message, k=RETRIEVAL_K)
        retrieved_context = [doc.page_content for doc in documents]

        system_prompt = self.system_prompt(retrieved_context)

        greetings = {
            "hi", "hello", "good day", "hey",
            "good morning", "good afternoon", "good evening"
        }
        normalized_message = message.lower().strip(" !?,.")
        if normalized_message in greetings:
            system_prompt = system_prompt.replace("Never introduce yourself unless appropriate.", "")

        history.append({"role": "user", "content": message})
        messages = [{"role": "system", "content": system_prompt}] + history

        outputs = self.pipeline(
            messages,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            return_full_text=True,
        )

        generated = outputs[0]["generated_text"]

        # Transformers may return a list of chat messages for chat-style generation.
        if isinstance(generated, list):
            assistant_messages = [
                item for item in generated
                if isinstance(item, dict) and item.get("role") == "assistant"
            ]
            if not assistant_messages:
                raise RuntimeError("The model did not return an assistant response.")
            response_text = assistant_messages[-1].get("content", "").strip()
        else:
            # Fallback for text-style pipeline output.
            response_text = str(generated).strip()

        if not response_text:
            raise RuntimeError("The model returned an empty response.")

        history.append({"role": "assistant", "content": response_text})
        self.save_history(user_id, history)

        return response_text

## 9. Configure the Digiwise System Prompt

The system prompt defines the chatbot's role, communication style, scope, and interaction principles. It is intentionally designed to make digital literacy guidance approachable for users with different levels of digital experience.

In [ ]:
prompt = """
Role: You are Digiwise, a knowledgeable and approachable digital literacy mentor. You specialize in helping individuals navigate the digital world with confidence, from using devices and understanding online safety to managing digital identities and using common digital tools. Digiwise is calm, patient, and avoids jargon, ensuring all users—regardless of their digital experience—feel empowered.

Objective: Provide clear, empathetic guidance that builds the user's confidence and ability in the digital world. Maintain a helpful, friendly tone and explain concepts in a way that is easy to understand. Encourage curiosity and offer reassurance when users feel overwhelmed or uncertain.

Guidelines:
- Clarity and Simplicity: Break down complex digital concepts into easy-to-understand explanations. Avoid technical jargon unless necessary, and explain it when used.
- Encouraging Mindset: Be reassuring, confidence-building, and judgment-free when users are uncertain about their digital skills.
- Dynamic Phrasing: Vary sentence openings and explanations rather than repeating the same wording.
- Supportive Tone: Do not rush users or belittle a lack of knowledge.
- Interactive Guidance: When appropriate, provide step-by-step instructions and ask useful follow-up questions.
- Stay Focused: Respond only to digital literacy-related topics. For unrelated requests, explain that you help with digital skills and online tools.
- Clarify When Unclear: Ask for clarification instead of making unsupported assumptions.
- Gentle Encouragement: When appropriate, suggest additional learning resources, including www.uniccongroup.com/digital.
- Avoid AI References: Speak as Digiwise, the digital mentor, rather than discussing the underlying AI system.

Fallback response:
“I’m not sure I fully understand yet, but I’d love to help you figure this out. Can you tell me more about what you’re trying to do?”

Never introduce yourself unless appropriate to the conversation.
""".strip()

## 10. Initialize the Chatbot

In [ ]:
digital_chatbot = DigiwiseChatbot(
    model=model,
    tokenizer=tokenizer,
    embedding=embeddings,
    prompt=prompt,
    vector_store_path=VECTOR_STORE_PATH,
)

print("Digiwise chatbot initialized successfully.")

## 11. Test the RAG Chatbot

Run a simple test before launching the interface. The example below mirrors the original notebook's communication-tools use case while using a clearer question.

In [ ]:
def chatbot(user_id, query):
    """Gradio-compatible wrapper around the Digiwise chatbot."""
    try:
        return digital_chatbot.chat_function(query, user_id)
    except Exception as exc:
        # Keep technical details out of the user-facing response.
        print(f"Chatbot error: {type(exc).__name__}: {exc}")
        return "Sorry, something went wrong while processing your request. Please try again."

response = chatbot("Colab_User1", "What tools can I use for online communication?")
print(response)

## 12. Launch the Gradio Interface

The interface below provides a simple browser-based demonstration.

`share=True` is intentionally disabled by default in this public-ready notebook. When running temporarily in Google Colab, you may set it to `True` if you need a temporary public share link. A temporary Gradio share link should not be treated as permanent hosting.

In [ ]:
import gradio as gr


def gradio_chat(user_id, message):
    """Handle messages submitted through the Gradio interface."""
    return chatbot(user_id, message)


demo = gr.Interface(
    fn=gradio_chat,
    inputs=[
        gr.Textbox(
            label="User ID",
            placeholder="Enter a simple user ID, e.g. User_001",
        ),
        gr.Textbox(
            label="Message",
            placeholder="Ask a digital literacy question...",
            lines=4,
        ),
    ],
    outputs=gr.Textbox(label="Digiwise"),
    title="Digiwise — Digital Literacy Chatbot",
    description=(
        "Ask Digiwise questions about digital skills, online tools, "
        "devices, online safety, and related digital literacy topics."
    ),
    examples=[
        ["Demo_User", "What tools can I use for online communication?"],
        ["Demo_User", "How can I stay safer when using public Wi-Fi?"],
        ["Demo_User", "What is two-factor authentication?"],
    ],
)

demo.launch(debug=False, share=False)

## 13. Limitations and Responsible Use

- **Knowledge-base dependency:** Retrieval quality depends on the documents contained in the ChromaDB vector store.
- **Model limitations:** The language model can produce incorrect or incomplete information. Responses should be reviewed when accuracy is important.
- **Hardware:** GPU acceleration is recommended for practical interactive performance.
- **Local conversation storage:** Conversation history is stored as JSON files under `USERS/`. This prototype does not provide production-grade authentication, encryption, access control, or database management.
- **Temporary demos:** Gradio's `share=True` option creates a temporary share link and is not a substitute for secure production hosting.
- **Data privacy:** Do not use this prototype with sensitive personal information unless appropriate privacy and security controls have been implemented.

## 14. Suggested GitHub Repository Structure

The notebook is intended to sit inside a repository that can grow beyond a single Colab file. A practical structure is:

```text
Digiwise-RAG-Chatbot/
├── README.md
├── Rag_ChatBot_llama.ipynb
├── requirements.txt
├── .gitignore
├── LICENSE
├── data/
│   └── README.md
├── assets/
│   ├── architecture.png
│   └── demo.png
└── USERS/              # local runtime data; do not commit
```

The exact repository structure can be simplified depending on whether the vector store and source documents are distributed with the project.

## 15. Reproducibility Checklist

Before publishing this notebook on GitHub:

- [ ] Run the notebook from a clean runtime using **Restart & Run All**.
- [ ] Confirm that no passwords, API keys, access tokens, or private data appear in any cell or output.
- [ ] Confirm that the ChromaDB vector store is either included, reproducibly generated, or clearly documented as an external prerequisite.
- [ ] Save the package dependencies in `requirements.txt`.
- [ ] Add `USERS/` and other local/runtime files to `.gitignore`.
- [ ] Remove unnecessary installation logs and development/debugging output before committing.
- [ ] Test the README instructions from a fresh environment if possible.